In [1]:
pip install selenium beautifulsoup4 pandas webdriver-manager


  Using cached selenium-4.36.0-py3-none-any.whl (9.6 MB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
  Using cached pandas-2.3.3-cp39-cp39-win_amd64.whl (11.4 MB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
  Using cached trio-0.31.0-py3-none-any.whl (512 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl (152 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl (15.9 MB)
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.w

You should consider upgrading via the 'c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv_kernel4\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
import concurrent.futures
import time

DEBUG_PORT = 9221

def try_connect_existing_chrome():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "debuggerAddress", f"127.0.0.1:{DEBUG_PORT}"
    )
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def get_or_create_driver(timeout=5):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            print("🔁 Tentative de connexion à Chrome existant...")
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(try_connect_existing_chrome)
                driver = future.result(timeout=timeout)
            print("✅ Connecté à Chrome existant")
            return driver
        except (WebDriverException, concurrent.futures.TimeoutError):
            print("⏳ Chrome non dispo ou timeout, retry...")
            time.sleep(0.5)

    # Après timeout → lancement d'un nouveau Chrome
    print("🚀 Timeout atteint → lancement d'un nouveau Chrome")
    options = webdriver.ChromeOptions()
    options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("🆕 Nouveau Chrome lancé avec debugging")
    return driver


In [2]:
def get_last_radix_buttons():
    """
    Récupère les boutons du dernier pop-up Radix ouvert.
    Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
    """

    # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
    radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
    if not radix_roots:

        return []

    # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
    radix_root = radix_roots[-1]

    # 3️⃣ le div interne qui contient les boutons
    try:
        radix_div = radix_root.find_element(By.XPATH, "./div")
    except:

        return []

    # 4️⃣ récupérer les boutons
    buttons = radix_div.find_elements(By.TAG_NAME, "button")

    return buttons

In [3]:
import time
from selenium.webdriver.common.by import By

def init_parse(driver, scroll_pause=1.0, scroll_step=500):

    champions_by_key = {}

    scroll_top = 0
    print("🚀 init_parse() démarré")

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container = driver.find_element(By.CSS_SELECTOR, container_selector)

        page_scroll_y = driver.execute_script("return window.pageYOffset;")
        table_scroll_y = driver.execute_script(
            "return arguments[0].scrollTop;", container
        )

        rows = container.find_elements(By.XPATH, "./div/div")

        for idx, row in enumerate(rows):
            text = row.text.strip()
            if not text:
                continue

            lines = text.splitlines()
            label = lines[0]
            name = lines[1] if len(lines) > 1 else "Unknown"

            # ✅ récupération URL
            url = None
            try:
                link = row.find_element(By.XPATH, ".//a")
                url = link.get_attribute("href")
            except:
                pass

            if url:
                print(f"🔗 {label} → {url}")

            key = text

            if key not in champions_by_key:
                champions_by_key[key] = {
                    "scroll_positions": [],
                    "label": label,
                    "numero": idx,
                    "name": name,
                    "url": url,   # ✅ stockée ici
                }

            champions_by_key[key]["scroll_positions"].append({
                "page": page_scroll_y,
                "table": table_scroll_y
            })

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")

        if scroll_top >= new_height:
            break

    champions = []

    for champ in champions_by_key.values():
        positions = champ["scroll_positions"]

        if len(positions) >= 2:
            p1, p2 = positions[-2], positions[-1]
            champ["scroll_page"] = (p1["page"] + p2["page"]) // 2
            champ["scroll_table"] = (p1["table"] + p2["table"]) // 2
        else:
            champ["scroll_page"] = positions[-1]["page"]
            champ["scroll_table"] = positions[-1]["table"]

        del champ["scroll_positions"]
        champions.append(champ)

    print(f"✅ {len(champions)} champions collectés (scroll + url)")
    return champions


In [4]:
import hashlib
from selenium.webdriver.common.by import By

ROLE_HASH_TO_TEXT = {
    "df14d23b35c9842bd8afa0db2b922aaf": "top",
    "bd9e54c883010f9bc2487e7d26a91b77": "jun",
    "3ce111209b6d69bee8498e94b02567ad": "mid",
    "6f1d8859e29002c2c45b527544e5a755": "adc",
    "3b66426a9b2beeca218cc726985f68b1": "sup",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role

# svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()

def hash_svg_path(svg_element) -> str:
    """
    Extrait le path 'd' du SVG et retourne son hash MD5
    """
    paths = svg_element.find_elements(By.TAG_NAME, "path")
    if not paths:
        return None

    path_d = paths[0].get_attribute("d").strip()
    return hashlib.md5(path_d.encode("utf-8")).hexdigest()


In [ ]:
# def get_selected_played_lane(driver) -> dict:
#     print("🟢 Détection de la played lane sélectionnée")

#     # 1️⃣ on récupère TOUS les containers possibles (sécurise si la page évolue)
#     containers = driver.find_elements(
#         By.CSS_SELECTOR,
#         "div.border-black-600.flex.w-fit.items-center.justify-center"
#     )

#     print(f"🔍 {len(containers)} containers candidats trouvés")

#     if not containers:
#         print("❌ Aucun container trouvé dans get_selected_played_lane")
#         return {"lane": None, "percentage": None}

#     # 2️⃣ on prend le premier (structure unique sur la page)
#     container = containers[0]

#     lane_divs = container.find_elements(By.XPATH, "./div")
#     print(f"🔍 {len(lane_divs)} boutons de lane trouvés")

#     # 3️⃣ on parcourt uniquement les 5 boutons
#     for idx, lane_div in enumerate(lane_divs):
#         try:
#             link = lane_div.find_element(By.TAG_NAME, "a")
#             inner_div = link.find_element(By.TAG_NAME, "div")

#             class_name = inner_div.get_attribute("class") or ""
#             print(f"[Lane {idx}] classes = {class_name}")

#             # pas sélectionné → on skip
#             if "bg-blue-200" not in class_name:
#                 continue

#             print(f"⭐ Lane sélectionnée détectée (index {idx})")

#             # SVG → hash → lane
#             svg = inner_div.find_element(By.TAG_NAME, "svg")
#             svg_hash = hash_svg_path(svg)
#             lane = resolve_role_from_hash(svg_hash)

#             # print(f"   🔐 svg_hash = {svg_hash}")
#             print(f"   🏷️ lane     = {lane}")

#             # 4️⃣ span JUSTE APRÈS le <a>
#             percentage = None
#             try:
#                 span = lane_div.find_element(By.TAG_NAME, "span")
#                 percentage = span.text.strip()
#             except Exception:
#                 print("⚠️ Span pourcentage introuvable")

#             print(f"   📊 percentage = {percentage}")

#             return {
#                 "lane": lane,
#                 "percentage": percentage
#             }

#         except Exception as e:
#             print("⚠️ ❌ Erreur lors de la détection de la lane sélectionnée")
#             print(f"[Lane {idx}] ❌ erreur : {e}")

#     print("❌ Aucune lane sélectionnée trouvée")
#     return {"lane": None, "percentage": None}


In [ ]:
# import time
# from selenium.common.exceptions import (
#     StaleElementReferenceException,
#     NoSuchElementException,
#     WebDriverException
# )

# def get_selected_played_lane(driver) -> dict:
#     print("🟢 Détection de la played lane sélectionnée")

#     # 1️⃣ on récupère TOUS les containers possibles (sécurise si la page évolue)
#     containers = driver.find_elements(
#         By.CSS_SELECTOR,
#         "div.border-black-600.flex.w-fit.items-center.justify-center"
#     )

#     print(f"🔍 {len(containers)} containers candidats trouvés")

#     if not containers:
#         print("❌ Aucun container trouvé dans get_selected_played_lane")
#         return {"lane": None, "percentage": None}

#     # 2️⃣ on prend le premier (structure unique sur la page)
#     container = containers[0]

#     lane_divs = container.find_elements(By.XPATH, "./div")
#     print(f"🔍 {len(lane_divs)} boutons de lane trouvés")

#     # 3️⃣ on parcourt uniquement les 5 boutons
#     for idx, lane_div in enumerate(lane_divs):

#         for attempt in range(1, 10):

#             try:
#                 # print(f"\n🔁 Lane {idx} — tentative {attempt}/2")

#                 link = lane_div.find_element(By.TAG_NAME, "a")
#                 inner_div = link.find_element(By.TAG_NAME, "div")

#                 class_name = inner_div.get_attribute("class") or ""
#                 print(f"[Lane {idx}] classes = {class_name}")

#                 # pas sélectionné → skip sans retry inutile
#                 if "bg-blue-200" not in class_name:
#                     break

#                 print(f"⭐ Lane sélectionnée détectée (index {idx})")

#                 # SVG → hash → lane
#                 svg = inner_div.find_element(By.TAG_NAME, "svg")
#                 svg_hash = hash_svg_path(svg)
#                 lane = resolve_role_from_hash(svg_hash)

#                 print(f"🏷️🏷️🏷️🏷️🏷️🏷️ lane = {lane}")

#                 # # span pourcentage
#                 # percentage = None
#                 # try:
#                 #     span = lane_div.find_element(By.TAG_NAME, "span")
#                 #     percentage = span.text.strip()
#                 #     print(f"🏷️🏷️🏷️🏷️🏷️🏷️ pourcentage = {percentage}")
#                 # # except NoSuchElementException:
#                 # except Exception:
#                 #     print("⚠️ Span pourcentage introuvable")

#                  # span pourcentage
#                 percentage = None
#                 span = lane_div.find_element(By.TAG_NAME, "span")
#                 percentage = span.text.strip()
#                 print(f"🏷️🏷️🏷️🏷️🏷️🏷️ pourcentage = {percentage}")


#                 #  # span pourcentage
#                 # percentage = None
#                 # try:
#                 #     span = lane_div.find_element(By.TAG_NAME, "span")
#                 #     percentage = span.text.strip()
#                 #     print(f"🏷️🏷️🏷️🏷️🏷️🏷️ pourcentage = {percentage}")
#                 # # except NoSuchElementException:
#                 # except Exception:
#                 #     print("⚠️ Span pourcentage introuvable")

#                 print(f"  📊📊📊📊 📊 percentage = {percentage}")

#                 return {
#                     "lane": lane,
#                     "percentage": percentage
#                 }

#             except (
#                 StaleElementReferenceException,
#                 NoSuchElementException,
#                 WebDriverException
#             ) as e:

#                 print(f"⚠️ Tentative de detection de lane {attempt} échouée lane {idx} → {type(e).__name__}")

#                 # if attempt == 2:
#                 #     print("❌ Retry max atteint — abandon lane")
#                 # else:
#                 #     time.sleep(0.5)
#                 time.sleep(3)

#     print("❌❌❌❌❌❌❌❌❌❌❌❌ Aucune lane sélectionnée trouvée ou aucun poucentage trouvé en 10 essais")
#     return {"lane": None, "percentage": None}


In [5]:
import time
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    WebDriverException
)

def get_selected_played_lane(driver) -> dict:
    print("\n" + "="*60)
    print("🟢 Détection de la played lane sélectionnée")
    print("="*60)

    MAX_RETRY = 10
    WAIT_SECONDS = 3

    for attempt in range(1, MAX_RETRY + 1):

        print(f"\n🔁 PASS GLOBAL {attempt}/{MAX_RETRY}")

        try:
            containers = driver.find_elements(
                By.CSS_SELECTOR,
                "div.border-black-600.flex.w-fit.items-center.justify-center"
            )

            print(f"🔍 {len(containers)} containers candidats trouvés")

            if not containers:
                raise NoSuchElementException("container introuvable")

            container = containers[0]
            lane_divs = container.find_elements(By.XPATH, "./div")

            print(f"🔍 {len(lane_divs)} boutons de lane trouvés")

            for idx, lane_div in enumerate(lane_divs):

                link = lane_div.find_element(By.TAG_NAME, "a")
                inner_div = link.find_element(By.TAG_NAME, "div")

                class_name = inner_div.get_attribute("class") or ""
                print(f"[Lane {idx}] classes = {class_name}")

                # pas sélectionné → skip direct
                if "bg-blue-200" not in class_name:
                    continue

                print(f"⭐ Lane sélectionnée détectée (index {idx})")

                svg = inner_div.find_element(By.TAG_NAME, "svg")
                svg_hash = hash_svg_path(svg)
                lane = resolve_role_from_hash(svg_hash)

                print(f"🏷️🏷️🏷️ lane = {lane}")

                span = lane_div.find_element(By.TAG_NAME, "span")
                percentage = span.text.strip()

                print(f"📊📊📊 percentage = {percentage}")

                if percentage and percentage != "-":
                    print("🟢 RESULTAT VALIDE")
                    return {
                        "lane": lane,
                        "percentage": percentage
                    }

                print("⚠️ percentage invalide → retry")

        except (
            StaleElementReferenceException,
            NoSuchElementException,
            WebDriverException
        ) as e:

            print(f"❌ Erreur tentative {attempt} → {type(e).__name__}")

        # -----------------------------
        # REFRESH TOUS LES 2 RETRIES
        # -----------------------------

        if attempt % 2 == 0 and attempt < MAX_RETRY:
            print("\n🔄🔄🔄 REFRESH PAGE (tous les 2 retry)")
            driver.refresh()
            time.sleep(WAIT_SECONDS)

        else:
            time.sleep(WAIT_SECONDS)

    print("\n" + "❌"*20)
    print("❌ Aucune lane sélectionnée trouvée après 10 tentatives")
    print("❌"*20)

    return {"lane": None, "percentage": None}


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException

# def collect_champion_lane_stats(driver) -> dict:
#     print("🟢 STATS DE LANE ---  Collecte des stats champion / lane")

#     stats = {
#         "tier": None,
#         "rank": None,
#         "winrate": None,
#         "pickrate": None,
#         "banrate": None,
#         "games": None,
#     }

#     try:
#         container = driver.find_element(
#             By.XPATH,
#             '//*[@id="root"]/main/div/div[3]/div[2]/div/div[2]/div[1]'
#         )
#     except NoSuchElementException:
#         print("❌ Container stats introuvable")
#         return stats

#     print("✅ Container stats trouvé")
#     print("🎨 class :", container.get_attribute("class"))

#     spans = container.find_elements(By.TAG_NAME, "span")
#     print(f"🔢 {len(spans)} spans trouvés\n")

#     KNOWN_KEYS = {
#         "niveau": "tier",
#         "rang": "rank",
#         "winrate": "winrate",
#         "pickrate": "pickrate",
#         "banrate": "banrate",
#         "parties": "games",
#     }

#     for idx, span in enumerate(spans):
#         raw_text = span.text.strip()
#         if not raw_text:
#             continue

#         # print(f"[Span {idx}] → '{raw_text}'")

#         lower = raw_text.lower()

#         for label, key in KNOWN_KEYS.items():
#             if label in lower:
#                 # ne pas écraser une valeur déjà trouvée
#                 if stats[key] is not None:
#                     continue

#                 # extraction valeur
#                 value = (
#                     raw_text
#                     .replace(label, "")
#                     .replace("\n", " ")
#                     .strip()
#                 )

#                 # ignorer les spans "label only"
#                 if not value:
#                     continue

#                 # suppression de l'espace insécable pour "games"
#                 if key == "games":
#                     value = value.replace("\u202f", "").replace(" ", "")

#                 stats[key] = value
#                 # print(f"✅ {key} détecté → '{value}'")

#     print("✅ Stats collectées :", stats)
#     return stats


In [6]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import time


def collect_champion_lane_stats(driver) -> dict:
    print("collect_champion_lane_stats 🟢 STATS DE LANE --- Collecte des stats champion / lane")

    XPATH = '//*[@id="root"]/main/div/div[3]/div[2]/div/div[2]/div[1]'

    KNOWN_KEYS = {
        "niveau": "tier",
        "rang": "rank",
        "winrate": "winrate",
        "pickrate": "pickrate",
        "banrate": "banrate",
        "parties": "games",
    }

    for attempt in range(1, 4):  # ✅ 3 tentatives

        print(f"collect_champion_lane_stats \n🔁 Tentative stats {attempt}/3")

        stats = {
            "tier": None,
            "rank": None,
            "winrate": None,
            "pickrate": None,
            "banrate": None,
            "games": None,
        }

        try:
            container = driver.find_element(By.XPATH, XPATH)
            print("collect_champion_lane_stats ✅ Container stats trouvé")
            print("collect_champion_lane_stats 🎨 class :", container.get_attribute("class"))

            spans = container.find_elements(By.TAG_NAME, "span")
            print(f"collect_champion_lane_stats 🔢 {len(spans)} spans trouvés")

        except (NoSuchElementException, StaleElementReferenceException) as e:
            print(f"collect_champion_lane_stats ❌ Container introuvable — {type(e).__name__}")

            if attempt < 3:
                print("collect_champion_lane_stats⏳ Attente 2 secondes avant retry…")
                time.sleep(2)
                continue
            else:
                print("collect_champion_lane_stats ❌ Abandon stats (container jamais trouvé)")
                return stats

        # -------------------------
        # Extraction
        # -------------------------

        for span in spans:
            raw_text = span.text.strip()
            if not raw_text:
                continue

            lower = raw_text.lower()

            for label, key in KNOWN_KEYS.items():
                if label in lower and stats[key] is None:

                    value = (
                        raw_text
                        .replace(label, "")
                        .replace("\n", " ")
                        .strip()
                    )

                    if not value:
                        continue

                    if key == "games":
                        value = value.replace("\u202f", "").replace(" ", "")

                    stats[key] = value

        # -------------------------
        # Validation
        # -------------------------

        found_count = sum(v is not None for v in stats.values())
        print(f"collect_champion_lane_stats📊 Stats détectées = {found_count}/6")

        if found_count >= 4:  # seuil raisonnable
            print("✅✅✅✅✅collect_champion_lane_stats✅ Stats collectées :", stats)
            print(stats)
            return stats

        print("collect_champion_lane_stats⚠️ Stats incomplètes — retry nécessaire")

        if attempt < 3:
            time.sleep(2)

    print("collect_champion_lane_stats ❌ Stats non récupérées après retries")
    return stats


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
# import time


# def collect_champion_lane_stats(driver) -> dict:
#     print("\n" + "=" * 70)
#     print("collect_champion_lane_stats 🟢 STATS DE LANE — START")
#     print("=" * 70)

#     XPATH = '//*[@id="root"]/main/div/div[3]/div[2]/div/div[2]/div[1]'

#     KNOWN_KEYS = {
#         "niveau": "tier",
#         "rang": "rank",
#         "winrate": "winrate",
#         "pickrate": "pickrate",
#         "banrate": "banrate",
#         "parties": "games",
#     }

#     MAX_RETRY = 10
#     WAIT_SECONDS = 3

#     for attempt in range(1, MAX_RETRY + 1):

#         print("\n" + "-" * 60)
#         print(f"collect_champion_lane_stats 🔁 TENTATIVE {attempt}/{MAX_RETRY}")
#         print("-" * 60)

#         stats = {
#             "tier": None,
#             "rank": None,
#             "winrate": None,
#             "pickrate": None,
#             "banrate": None,
#             "games": None,
#         }

#         # -------------------------
#         # FIND CONTAINER
#         # -------------------------

#         try:
#             container = driver.find_element(By.XPATH, XPATH)
#             spans = container.find_elements(By.TAG_NAME, "span")

#             print("collect_champion_lane_stats ✅ Container trouvé")
#             print(f"collect_champion_lane_stats 🔢 spans = {len(spans)}")

#         except (NoSuchElementException, StaleElementReferenceException) as e:
#             print(f"collect_champion_lane_stats ❌ Container erreur → {type(e).__name__}")

#             if attempt < MAX_RETRY:
#                 print(f"collect_champion_lane_stats ⏳ retry dans {WAIT_SECONDS}s")
#                 time.sleep(WAIT_SECONDS)
#                 continue
#             else:
#                 print("collect_champion_lane_stats ❌ ECHEC TOTAL container")
#                 return stats

#         # -------------------------
#         # EXTRACTION
#         # -------------------------

#         for span in spans:
#             raw_text = span.text.strip()
#             if not raw_text:
#                 continue

#             lower = raw_text.lower()

#             for label, key in KNOWN_KEYS.items():

#                 if label in lower and stats[key] is None:

#                     value = (
#                         raw_text
#                         .replace(label, "")
#                         .replace("\n", " ")
#                         .strip()
#                     )

#                     if not value or value == "-" or value == "-/-":
#                         continue

#                     if key == "games":
#                         value = value.replace("\u202f", "").replace(" ", "")

#                     stats[key] = value

#         # -------------------------
#         # VALIDATION
#         # -------------------------

#         valid_keys = [k for k, v in stats.items() if v not in (None, "-", "")]
#         missing_keys = [k for k, v in stats.items() if v in (None, "-", "")]

#         print("\ncollect_champion_lane_stats 📊 ETAT ACTUEL")
#         for k, v in stats.items():
#             print(f"   {k:8} → {v}")

#         print("\ncollect_champion_lane_stats ✅ trouvés :", len(valid_keys), "/ 6")

#         if missing_keys:
#             print("collect_champion_lane_stats ❌ MANQUANTS →")
#             for m in missing_keys:
#                 print(f"   ❌ {m}")

#         # -------------------------
#         # SUCCESS CONDITION
#         # -------------------------

#         if len(valid_keys) == 6:
#             print("\n" + "🟢" * 20)
#             print("collect_champion_lane_stats ✅✅✅ STATS COMPLETES")
#             print(stats)
#             print("🟢" * 20 + "\n")
#             return stats

#         # -------------------------
#         # RETRY
#         # -------------------------

#         if attempt < MAX_RETRY:
#             print(f"\ncollect_champion_lane_stats ⏳ retry dans {WAIT_SECONDS}s (valeurs '-' ou manquantes)")
#             time.sleep(WAIT_SECONDS)

#     # -------------------------
#     # FAIL AFTER RETRIES
#     # -------------------------

#     print("\n" + "🔴" * 20)
#     print("collect_champion_lane_stats ❌ ECHEC APRES 10 RETRIES")
#     print(stats)
#     print("🔴" * 20 + "\n")

#     return stats


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
# import time


# def collect_champion_lane_stats(driver) -> dict:
#     print("\n" + "=" * 70)
#     print("collect_champion_lane_stats 🟢 STATS DE LANE — START")
#     print("=" * 70)

#     XPATH = '//*[@id="root"]/main/div/div[3]/div[2]/div/div[2]/div[1]'

#     KNOWN_KEYS = {
#         "niveau": "tier",
#         "rang": "rank",
#         "winrate": "winrate",
#         "pickrate": "pickrate",
#         "banrate": "banrate",
#         "parties": "games",
#     }

#     MAX_RETRY = 10
#     WAIT_SECONDS = 3

#     for attempt in range(1, MAX_RETRY + 1):

#         print("\n" + "-" * 60)
#         print(f"collect_champion_lane_stats 🔁 TENTATIVE {attempt}/{MAX_RETRY}")
#         print("-" * 60)

#         stats = {
#             "tier": None,
#             "rank": None,
#             "winrate": None,
#             "pickrate": None,
#             "banrate": None,
#             "games": None,
#         }

#         # -------------------------
#         # FIND CONTAINER
#         # -------------------------

#         try:
#             container = driver.find_element(By.XPATH, XPATH)
#             spans = container.find_elements(By.TAG_NAME, "span")

#             print("collect_champion_lane_stats ✅ Container trouvé")
#             print(f"collect_champion_lane_stats 🔢 spans = {len(spans)}")

#         except (NoSuchElementException, StaleElementReferenceException) as e:
#             print(f"collect_champion_lane_stats ❌ Container erreur → {type(e).__name__}")

#             if attempt < MAX_RETRY:
#                 print("🔄 Refresh page + retry")
#                 driver.refresh()
#                 time.sleep(WAIT_SECONDS)
#                 continue
#             else:
#                 print("collect_champion_lane_stats ❌ ECHEC TOTAL container")
#                 return stats

#         # -------------------------
#         # EXTRACTION
#         # -------------------------

#         for span in spans:
#             raw_text = span.text.strip()
#             if not raw_text:
#                 continue

#             lower = raw_text.lower()

#             for label, key in KNOWN_KEYS.items():

#                 if label in lower and stats[key] is None:

#                     value = (
#                         raw_text
#                         .replace(label, "")
#                         .replace("\n", " ")
#                         .strip()
#                     )

#                     if not value or value == "-":
#                         continue

#                     if key == "games":
#                         value = value.replace("\u202f", "").replace(" ", "")

#                     stats[key] = value

#         # -------------------------
#         # VALIDATION
#         # -------------------------

#         valid_keys = [k for k, v in stats.items() if v not in (None, "-", "")]
#         missing_keys = [k for k, v in stats.items() if v in (None, "-", "")]

#         print("\ncollect_champion_lane_stats 📊 ETAT ACTUEL")
#         for k, v in stats.items():
#             print(f"   {k:8} → {v}")

#         print("\ncollect_champion_lane_stats ✅ trouvés :", len(valid_keys), "/ 6")

#         if missing_keys:
#             print("collect_champion_lane_stats ❌ MANQUANTS →")
#             for m in missing_keys:
#                 print(f"   ❌ {m}")

#         # -------------------------
#         # SUCCESS
#         # -------------------------

#         if len(valid_keys) == 6:
#             print("\n" + "🟢" * 20)
#             print("collect_champion_lane_stats ✅✅✅ STATS COMPLETES")
#             print(stats)
#             print("🟢" * 20 + "\n")
#             return stats

#         # -------------------------
#         # RETRY WITH REFRESH
#         # -------------------------

#         if attempt < MAX_RETRY:
#             print(f"\n🔄 Refresh page + retry dans {WAIT_SECONDS}s")
#             driver.refresh()
#             time.sleep(WAIT_SECONDS)

#     # -------------------------
#     # FAIL
#     # -------------------------

#     print("\n" + "🔴" * 20)
#     print("collect_champion_lane_stats ❌ ECHEC APRES 10 RETRIES")
#     print(stats)
#     print("🔴" * 20 + "\n")

#     return stats


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException, WebDriverException
# import time

# def click_matchup_or_synergy(driver, matchup: bool) -> bool:
#     target_name = "MATCHUPS" if matchup else "SYNERGIES"
#     aria_key = "Enemy" if matchup else "Ally"

#     print(f"🟢 Sélection de l'onglet {target_name} ---------------------- CMICK MATCHUP", flush=True)

#     xpath = f"//button[@role='tab' and contains(@aria-controls, '{aria_key}')]"

#     try:
#         button = driver.find_element(By.XPATH, xpath)

#         data_state = button.get_attribute("data-state")
#         print(f"🔍 {target_name} data-state = {data_state}")

#         if data_state == "active":
#             print(f"ℹ️ Onglet {target_name} déjà actif")
#             return False

#         driver.execute_script(
#             "arguments[0].scrollIntoView({block: 'center'});",
#             button
#         )
#         time.sleep(0.3)

#         button.click()
#         print(f"🖱️ Onglet {target_name} cliqué", flush=True)
#         time.sleep(0.7)

#         return True

#     except NoSuchElementException:
#         print(f"❌ Onglet {target_name} introuvable (aria-controls)", flush=True)
#         print(f"❌ ❌ ❌ ❌ ❌ probleme click_matchup_or_synergy", flush=True)
#         return False

#     except WebDriverException as e:
#         print(f"❌ Erreur Selenium sur {target_name} : {e}", flush=True)
#         print(f"❌ ❌ ❌ ❌ ❌ probleme click_matchup_or_synergy", flush=True)
#         return False


In [7]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException,
    WebDriverException,
    StaleElementReferenceException,
    ElementClickInterceptedException
)
import time


def click_matchup_or_synergy(driver, matchup: bool) -> bool:
    target_name = "MATCHUPS" if matchup else "SYNERGIES"
    aria_key = "Enemy" if matchup else "Ally"

    print(f"🟢 Sélection onglet {target_name} ----------------------", flush=True)

    xpath = f"//button[@role='tab' and contains(@aria-controls, '{aria_key}')]"

    for attempt in range(1, 4):  # ✅ 3 tentatives
        try:
            print(f"🔁 Tentative {attempt}/3 — recherche bouton {target_name}")

            button = driver.find_element(By.XPATH, xpath)

            data_state = button.get_attribute("data-state")
            print(f"🔍 {target_name} data-state = {data_state}")

            if data_state == "active":
                print(f"ℹ️ℹ️ℹ️ℹ️ℹ️ CLICK MATCHUPS OK Onglet {target_name} déjà actif")
                
                return False

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                button
            )
            time.sleep(0.3)

            button.click()
            print(f"🖱️🖱️🖱️🖱️🖱️ CLICK MATCHUPS OK Onglet {target_name} cliqué", flush=True)
            time.sleep(0.7)

            return True

        except (
            NoSuchElementException,
            StaleElementReferenceException,
            ElementClickInterceptedException,
            WebDriverException
        ) as e:

            print(f"⚠️ Tentative {attempt} échouée — {type(e).__name__}", flush=True)

            if attempt < 3:
                print("⏳ Attente 1 seconde avant retry…", flush=True)
                time.sleep(1)
            else:
                print(f"❌ Onglet {target_name} introuvable après 3 tentatives", flush=True)
                print(f"❌ ❌ ❌ ❌ ❌ probleme click_matchup_or_synergy", flush=True)
                print(f"Erreur finale: {e}", flush=True)
                return False


In [8]:
def click_matchup_or_synergy_lane(driver, lane_name) -> bool:
    print(f"🟢 Sélection de la lane {lane_name} via hash du PATH SVG")

    time.sleep(0.5)

    lane_container_xpath = (
        '//*[@id="root"]/main[1]/div[1]/div[3]/div[2]/div[1]/div[2]/div[4]'
        '/div[1]/div[1]/div[3]/div[1]'
    )

    for attempt in range(1, 4):  # 3 attempts
        try:
            print(f"🔁 Recherche lane_container — tentative {attempt}/3")
            lane_container = driver.find_element(By.XPATH, lane_container_xpath)
            print("✅ lane_container trouvé")
            break

        except (NoSuchElementException, StaleElementReferenceException) as e:
            print(f"⚠️ Tentative {attempt} échouée : {type(e).__name__}")

            if attempt < 3:
                print("⏳ Attente 1 seconde avant retry…")
                time.sleep(1)
            else:
                print(f"❌ Conteneur des lanes introuvable après 3 tentatives : {e}")
                print("❌ ❌ ❌ ❌ ❌ probleme click_matchup_or_synergy_lane", flush=True)
                return False

    lane_buttons = lane_container.find_elements(By.TAG_NAME, "button")
    print(f"🔍 {len(lane_buttons)} boutons de lane trouvés\n")

    for i, btn in enumerate(lane_buttons):
        svgs = btn.find_elements(By.TAG_NAME, "svg")
        if not svgs:
            continue

        svg = svgs[0]
        svg_hash = hash_svg_path(svg)

        if not svg_hash:
            continue

        role = resolve_role_from_hash(svg_hash)

        print(f"[Lane {i}] hash={svg_hash} role={role}")

        if role == lane_name:
            print(f"🎯 Bouton {lane_name} identifié — clic")

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                btn,
            )
            time.sleep(0.3)
            btn.click()
            print(f"🎯🎯🎯🎯🎯🎯🎯 lane matchup cliqué OK")

            # print(f"🖱️ Bouton {lane_name} cliqué avec succès")
            return True

    print(f"❌ Bouton {lane_name} non trouvé")
    return False


In [9]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
    ElementNotInteractableException,
)
import time


def click_liste_complete(driver) -> bool:
    print("🟢 Recherche du bouton '+ Liste complète' parmi tous les boutons...", flush=True)

    for attempt in range(2):
        print(f"🔁 Tentative {attempt + 1}/2")

        try:
            buttons = driver.find_elements(By.TAG_NAME, "button")
            print(f"🔍 {len(buttons)} boutons trouvés sur la page")

            for i, btn in enumerate(buttons):
                try:
                    btn_text = btn.text.strip()
                    btn_text_clean = " ".join(btn_text.split())

                    # XPath absolu (debug)
                    btn_xpath = driver.execute_script(
                        """
                        function getXPath(element) {
                            if (element.id !== '')
                                return '//*[@id="' + element.id + '"]';
                            if (element === document.body)
                                return '/html/body';

                            let ix = 0;
                            let siblings = element.parentNode.childNodes;
                            for (let i = 0; i < siblings.length; i++) {
                                let sibling = siblings[i];
                                if (sibling === element)
                                    return getXPath(element.parentNode) + '/' +
                                        element.tagName.toLowerCase() + '[' + (ix + 1) + ']';
                                if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
                                    ix++;
                            }
                        }
                        return getXPath(arguments[0]);
                        """,
                        btn,
                    )

                    # print(f"[{i}] → '{btn_text_clean}' | XPath: {btn_xpath}")

                    if "+ Liste complète" in btn_text_clean:
                        print("🎯Bouton '+ Liste complète' détecté, tentative de clic...")

                        driver.execute_script(
                            "arguments[0].scrollIntoView({block: 'center'});",
                            btn,
                        )
                        time.sleep(0.75)
                        btn.click()

                        print("🖱️🖱️🖱️🖱️🖱️🖱️ Bouton '+ Liste complète' cliqué avec succès", flush=True)
                        time.sleep(1)
                        return True  # ✅ succès immédiat

                except StaleElementReferenceException:
                    print(f"⚠️ Bouton [{i}] devenu obsolète (DOM mis à jour)")
                except Exception as e:
                    print(f"⚠️ Erreur sur le bouton [{i}] : {e}")

        except (
            NoSuchElementException,
            ElementClickInterceptedException,
            ElementNotInteractableException,
        ) as e:
            print(f"❌ Erreur lors de la tentative {attempt + 1} : {e}")

        time.sleep(0.5)

    print("❌ Bouton '+ Liste complète' non cliqué après 2 tentatives", flush=True)
    return False  # ❌ échec final


In [10]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import time

def collect_matchup(driver, matchup: bool = True):
    """
    Parcourt les matchups (Enemy) ou synergies (Ally) après clic sur 'Liste complète'
    et retourne une liste de dicts :
    {
        champ_counter_i,
        winrate,
        games,
        lane_quality
    }

    :param matchup: True pour matchups (Enemy), False pour synergies (Ally)
    """

    print("🟢 Démarrage collect_matchup")
    results = []
    seen = set()

    # ===============================
    # 1️⃣ Section Enemy / Ally
    # ===============================
    aria_key = "Enemy" if matchup else "Ally"
    container = None
    for attempt in range(2):
        try:
            # on sélectionne le container via role="tabpanel" et aria-labelledby
            container = driver.find_element(
                By.XPATH,
                f"//div[@role='tabpanel' and contains(@aria-labelledby, '{aria_key}')]"
            )
            print(f"✅ Section {'Enemy' if matchup else 'Ally'} trouvée (aria-labelledby: {aria_key})")
            break
        except NoSuchElementException:
            print(f"⏳ Tentative {attempt+1}/2 : Section {'Enemy' if matchup else 'Ally'} introuvable")
            time.sleep(0.8)

    if container is None:
        print(f"❌ Section {'Enemy' if matchup else 'Ally'} introuvable après 2 tentatives !")
        return results

    # ===============================
    # 2️⃣ Récupérer les deux wrappers : visible + hors écran
    # ===============================
    try:
        visible_wrapper = container.find_element(By.XPATH, "./div/div[1]/div")
        hidden_wrapper = container.find_element(By.XPATH, "./div/div[2]/div")
        print("✅ Wrappers visible et hidden trouvés")
    except NoSuchElementException:
        print("❌ Wrappers introuvables !")
        return results

    # ===============================
    # 3️⃣ Fonction de parsing des cards
    # ===============================
    def parse_cards(wrapper):
        cards = wrapper.find_elements(By.XPATH, "./div")
        print(f"🔍 {len(cards)} cards trouvées dans ce wrapper")
        for idx, card in enumerate(cards):
            try:
                # ---------- <a> ----------
                anchor = card.find_element(By.XPATH, "./a")
                img = anchor.find_element(By.XPATH, ".//img")
                champ_name = img.get_attribute("alt").strip()
                if not champ_name or champ_name in seen:
                    continue
                seen.add(champ_name)

                # -------- winrate et nombre de parties --------
                info_div = anchor.find_element(By.XPATH, "./div/div[2]")  # div C
                spans = info_div.find_elements(By.XPATH, "./span")
                winrate = spans[0].text.strip() if len(spans) > 0 else ""
                games = spans[1].text.strip().replace("\u202f", "") if len(spans) > 1 else ""

                # -------- lane_quality --------
                lane_quality = ""
                try:
                    button = card.find_element(By.XPATH, "./button")
                    lane_quality = button.text.strip()
                except NoSuchElementException:
                    pass

                # print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

                results.append({
                    "champ_counter_i": champ_name,
                    "winrate": winrate,
                    "games": games,
                    "lane_quality": lane_quality
                })

            except StaleElementReferenceException:
                print("⚠️ StaleElement — skip")
            except Exception as e:
                print(f"❌ Erreur card {idx} → {e}")
        print("collect matchups 🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯")
        print(" | ".join(
            f"{r['champ_counter_i']} {r['winrate']} {r['games']} {r['lane_quality']}"
            for r in results
        ))
    # ===============================
    # 4️⃣ Parser les cards visibles et hors écran
    # ===============================
    parse_cards(visible_wrapper)
    parse_cards(hidden_wrapper)

    print(f"📦 {'Matchups' if matchup else 'Synergies'} collectés : {len(results)}")
    return results


In [11]:
import pandas as pd


def generate_csv_from_champions(
    all_champions_data,
    elo,
    server,
    patch,
    lane_inspected,
    synergy_or_matchup,
    output_path="matchups_champions.csv",
):
    """
    all_champions_data = [
        {
            "name": str,
            "role": str,
            "role_play_ratio": str,
            "tier": str,
            "rank": str,
            "winrate": str,
            "pickrate": str,
            "banrate": str,
            "nb_games_analyzed": str,
            "url": str,
            "collect_coherente": bool,

            "matchups": {
                "top": [ {...}, ... ],
                "jun": [ {...}, ... ],
                "mid": [ {...}, ... ],
                "adc": [ {...}, ... ],
                "sup": [ {...}, ... ],
            },

            "synergies": {
                "top": [ {...}, ... ],
                "jun": [ {...}, ... ],
                "mid": [ {...}, ... ],
                "adc": [ {...}, ... ],
                "sup": [ {...}, ... ],
            }
        },
        ...
    ]
    """
    print("génération du csv..................")

    lanes = ["top", "jun", "mid", "adc", "sup"]

    # ==========================================================
    # 1️⃣ Calculer les max par TYPE (matchup/synergy) et par LANE
    # ==========================================================
    max_matchups = {lane: 0 for lane in lanes}
    max_synergies = {lane: 0 for lane in lanes}

    for champ in all_champions_data:
        for lane in lanes:
            max_matchups[lane] = max(
                max_matchups[lane],
                len(champ.get("matchup", {}).get(lane, []))
            )
            max_synergies[lane] = max(
                max_synergies[lane],
                len(champ.get("synergy", {}).get(lane, []))
            )

    print("📊 Max matchups par lane :", max_matchups)
    print("📊 Max synergies par lane :", max_synergies)

    # ======================
    # 2️⃣ Construire les rows
    # ======================
    rows = []

    for champ in all_champions_data:
        row = {
            # -------- Colonnes principales (EN PREMIER) --------
            "champion": champ.get("name"),
            "role": champ.get("role"),
            "role_play_ratio": champ.get("role_play_ratio"),
            "tier": champ.get("tier"),
            "rank": champ.get("rank"),
            "winrate": champ.get("winrate"),
            "pickrate": champ.get("pickrate"),
            "banrate": champ.get("banrate"),
            "nb_games_analyzed": champ.get("nb_games_analyzed"),
            "url": champ.get("url"),
            "collect_coherente": champ.get("collect_coherente"),
        }

        # ======================
        # 3️⃣ MATCHUPS
        # ======================
        for lane in lanes:
            lane_matchups = champ.get("matchup", {}).get(lane, [])

            for i in range(max_matchups[lane]):
                prefix = f"matchup_{lane}_{i+1}"

                if i < len(lane_matchups):
                    m = lane_matchups[i]
                    row[f"{prefix}_name"] = m.get("champ_counter_i")
                    row[f"{prefix}_winrate"] = m.get("winrate")
                    row[f"{prefix}_games"] = m.get("games")
                    row[f"{prefix}_lane_quality"] = m.get("lane_quality")
                else:
                    row[f"{prefix}_name"] = None
                    row[f"{prefix}_winrate"] = None
                    row[f"{prefix}_games"] = None
                    row[f"{prefix}_lane_quality"] = None

        # ======================
        # 4️⃣ SYNERGIES
        # ======================
        for lane in lanes:
            lane_synergies = champ.get("synergy", {}).get(lane, [])

            for i in range(max_synergies[lane]):
                prefix = f"synergy_{lane}_{i+1}"

                if i < len(lane_synergies):
                    s = lane_synergies[i]
                    row[f"{prefix}_name"] = s.get("champ_counter_i")
                    row[f"{prefix}_winrate"] = s.get("winrate")
                    row[f"{prefix}_games"] = s.get("games")
                    row[f"{prefix}_lane_quality"] = s.get("lane_quality")
                else:
                    row[f"{prefix}_name"] = None
                    row[f"{prefix}_winrate"] = None
                    row[f"{prefix}_games"] = None
                    row[f"{prefix}_lane_quality"] = None

        rows.append(row)

    # ======================
    # 5️⃣ DataFrame + CSV
    # ======================
    df = pd.DataFrame(rows)

    # filename = f"{output_path.replace('.csv', '')}{param_lane}_{param_synergy_or_matchup}_{elo}_{server}_{patch}.csv"
    filename = output_path
    df.to_csv(filename, index=False)

    print(f"✅ CSV généré : {filename}")
    print(f"📐 Shape DF : {df.shape}")


In [14]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [45]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
✅ Connecté à Chrome existant


In [16]:
sauvegarde_champs_en_cours = {}

In [25]:
# from selenium.webdriver.common.by import By
# import time

# import sys
# sys.stdout.flush()

# if __name__ == "__main__":

#     param_lane = "top"
#     param_synergy_or_matchup = "matchup"
#     # param_lane = "jun"
#     # param_synergy_or_matchup = "matchup"
#     # param_lane = "mid"
#     # param_synergy_or_matchup = "matchup"
#     # param_lane = "adc"
#     # param_synergy_or_matchup = "matchup"
#     # param_lane = "sup"
#     # param_synergy_or_matchup = "matchup"

#     # param_lane = "top"
#     # param_synergy_or_matchup = "synergy"
#     # param_lane = "jun"
#     # param_synergy_or_matchup = "synergy"
#     # param_lane = "mid"
#     # param_synergy_or_matchup = "synergy"
#     # param_lane = "adc"
#     # param_synergy_or_matchup = "synergy"
#     # param_lane = "sup"
#     # param_synergy_or_matchup = "synergy"
     
#     # Variables pour suivre la combinaison active
#     elo = None
#     server = None
#     patch = None

#     big_champions_url_dict = {}
#     big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"] = {}  # initialisation de la clé principale
#     # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"] = {}
#     root_key = f"{param_lane}_{param_synergy_or_matchup}"
#     if root_key not in sauvegarde_champs_en_cours:
#         sauvegarde_champs_en_cours[root_key] = {}

#     filters_container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
#         "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
#     )
#     filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)

#     print("✅ Containers trouvés")

#     # =========================
#     # 1️⃣ Ouvrir ELO et noter la liste des boutons
#     # =========================
#     filters_container.find_element(By.XPATH, ".//button[1]").click()
#     time.sleep(0.5)
#     elo_buttons = get_last_radix_buttons()
#     print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

#     # =========================
#     # 2️⃣ Ouvrir SERVER et noter la liste des boutons
#     # =========================
#     filters_container.find_element(By.XPATH, ".//button[2]").click()
#     time.sleep(0.5)
#     server_buttons = get_last_radix_buttons()
#     print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

#     # =========================
#     # 3️⃣ Ouvrir PATCH et noter la liste des boutons
#     # =========================
#     filters_container.find_element(By.XPATH, ".//div/button").click()
#     time.sleep(0.5)
#     patch_buttons = get_last_radix_buttons()
#     print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")
    

#     # =========================
#     # BOUCLE SUR TOUTES LES COMBINAISONS
#     # =========================


#     # de elo_départ à elo_max
#     # for i in range(numeloDepart | 0, max(AeloFin, len(elo_buttons))):
#     # de elo_départ à elo_fin
#     # for i in range(numeloDepart | 0, min(AelohFin, len(elo_buttons))): 
#     # for i in range(0,max(1, len(elo_buttons))):
#     # for i in range(0,min(2, len(elo_buttons))):
#     for i in range(max(1, len(elo_buttons)) - 1, -1, -1):
#         if not (i in (0, 1, 3, 13, 15)):
#                 continue  # 🔹 on skip les serveurs non désirés
        
#         # 🔁 réouvrir la dropdown ELO
#         filters_container.find_element(By.XPATH, ".//button[1]").click()
#         time.sleep(0.7)
#         elo_buttons = get_last_radix_buttons()
#         elo_btn = elo_buttons[i]
#         elo = elo_btn.text.strip()


#         elo_btn.click()
#         print(f"\n🎯 ELO [{i}] cliqué → {elo}")

#         big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"] = {}
#         # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"] = {}
#         sauvegarde_champs_en_cours.setdefault(root_key, {}).setdefault(elo, {})
        
#         time.sleep(1)
#         print("1")
#         time.sleep(1)
#         print("2")

#         # de server_départ à server_max
#         # for j in range(numServerDepart | 0, max(AServerFin, len(server_buttons))):
#         # de server_départ à server_fin
#         # for j in range(numServerDepart | 0, min(AServerFin, len(server_buttons))): 
#         # for j in range(7, min(8, len(server_buttons))):
#         #     if ( ((j >= 3) and (j <= 10) and (j != 4) and(j!=7)) ): #7 pour LAS pour les tests


#         for j in range(max(8, len(server_buttons)) - 1, -1, -1):
#             if ((j >= 3) and (j <= 11)):
#                 continue  # 🔹 on skip les serveurs non désirés
            
#             # 🔁 réouvrir la dropdown SERVER
#             filters_container.find_element(By.XPATH, ".//button[2]").click()
#             time.sleep(0.7)
#             server_buttons = get_last_radix_buttons()
#             server_btn = server_buttons[j]
#             server = server_btn.text.strip()

#             big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"] = {}
#             # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"] = {}
#             sauvegarde_champs_en_cours.setdefault(root_key, {}).setdefault(elo, {}).setdefault(server, {})

#             server_btn.click()
#             print(f"  🌍 SERVER [{j}] cliqué → {server}")
#             time.sleep(1)
#             print("1")
#             time.sleep(1)
#             print("2")
#             time.sleep(1)
#             print("3")
#             filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)


#             # de patch_départ à patch_max
#             # for k in range(numPatchDepart | 0, max(APatchFin, len(patch_buttons))):
#             # de patch_départ à patch_fin
#             # for k in range(numPatchDepart | 0, min(APatchFin, len(patch_buttons))): 
#             # for k in range(5, min(6, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
#             for k in range(3, max(5, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester   
#                 if ((k ==0 ) or (k == 1) or (k==2) or (k == 4) or (k == 5) or (k == 7) or (k == 8) ):
#                     continue  # 🔹 on skip les serveurs non désirés       
#                 # 🔁 réouvrir la dropdown PATCH
#                 filters_container.find_element(By.XPATH, ".//div/button").click()
#                 time.sleep(1)

#                 # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
#                 patch_buttons = get_last_radix_buttons()
#                 time.sleep(1)
#                 print("click sur les patchs")
#                 print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
#                 patch_btn = patch_buttons[k]

#                 # 🔹 cliquer sur le kème bouton
#                 patch = patch_btn.text.strip()
#                 patch_btn.click()
#                 time.sleep(0.7)

#                 print(f"    🧩 PATCH [{k}] cliqué → {patch}")

#                 # ✅ COMBINAISON ACTIVE
#                 print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
#                 time.sleep(1)
#                 print("1")
#                 time.sleep(1)
#                 print("2")
#                 time.sleep(1)
#                 print("3")
                
#                 filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
                

#                 champions = init_parse(driver)
#                 big_champions_url_dict[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"][f"{patch}"] = champions
#                 # sauvegarde_champs_en_cours[f"{param_lane}_{param_synergy_or_matchup}"][f"{elo}"][f"{server}"][f"{patch}"] = champions
#                 sauvegarde_champs_en_cours.setdefault(root_key, {}).setdefault(elo, {}).setdefault(server, {}).setdefault(patch, champions)
                
#                 driver.execute_script("window.scrollTo(0, arguments[0]);", 0)




In [ ]:
# def build_champion_dict(obj):
#     """
#     Transforme le JSON imbriqué en :
#     {
#         "lane_mode_elo_server_patch": [champions...],
#         ...
#     }
#     """

#     result = {}

#     def walk(node, path_keys):
#         # cas feuille = liste de champions
#         if isinstance(node, list):
#             if len(path_keys) == 4:
#                 lane_mode, elo, server, patch = path_keys
#                 key = f"{lane_mode}_{elo}_{server}_{patch}"
#                 result[key] = node
#                 print(f"✅ Liste champions trouvée → {key} ({len(node)} champions)")
#             return

#         if isinstance(node, dict):
#             for k, v in node.items():
#                 walk(v, path_keys + [k])

#     walk(obj, [])
#     return result


In [26]:
# champion_dict = build_champion_dict(sauvegarde_champs_en_cours)

# print(len(champion_dict))
# print(list(champion_dict.keys())[:5])
# for k, v in champion_dict.items():
#     print(k, len(v))

In [12]:
from selenium.webdriver.common.by import By
import time

def open_champion_in_new_tab(driver, champion_descriptor: dict, wait_seconds: int = 5, param_lane="a definir", param_synergy_or_matchup="a definir"):
    # open_champion_in_new_tab(driver, champ, wait_seconds=5, param_lane=param_lane, param_synergy_or_matchup=param_synergy_or_matchup)
    """
    Ouvre le champion dans un nouvel onglet à partir de l'URL stockée dans champion_descriptor.
    
    champion_descriptor = {
        "name": str,
        "url": str,
        "label": str,
        ...
    }
    """
    print("OPEN CHAMPION --- DEBUT DE LA METHODE")
    url = champion_descriptor.get("url")
    label = champion_descriptor.get("label", "Unknown")

    print("lane a inspecter:", param_lane)
    print("synergy ou matchup:", param_synergy_or_matchup)
    if not url:
        print(f"❌ Pas d'URL pour {label}, impossible d'ouvrir")
        return None
    
    main_window = driver.current_window_handle 
    windows_before = driver.window_handles 
    print(f"🪟 Onglets AVANT = {windows_before}")

    # print("début de fermeture des onglets superflus")
    # main_window = driver.current_window_handle
    # windows_before = driver.window_handles
    # print(f"🪟 Onglets AVANT = {windows_before}")
    # handles = driver.window_handles
    # print("🪟 Onglets AVANT =", handles)
    # if not handles:
    #     print("❌ Aucun onglet")
    #     return
    # rightmost = handles[-1]  # ← le plus à droite / dernier ouvert
    # for h in handles:
    #     if h != rightmost:
    #         driver.switch_to.window(h)
    #         driver.close()
    # driver.switch_to.window(rightmost)
    # print("✅ Onglets restants =", driver.window_handles)
    # print("fin de fermeture des onglets superflus")


    print(f"🌍 tentative d' Ouverture champion '{label}' dans un nouvel onglet → {url}")

    time.sleep(0.6)
    # Ouvrir nouvel onglet
    driver.execute_script(f"window.open('{url}', '_blank');")
    time.sleep(0.6)

    windows_after = driver.window_handles
    print(f"🪟 Onglets APRES = {windows_after}")

    # Identifier le nouvel onglet
    new_tabs = [w for w in windows_after if w not in windows_before]
    if not new_tabs:
        print("❌ Aucun nouvel onglet détecté")
        return None

    new_tab = new_tabs[0]
    driver.switch_to.window(new_tab)
    print(f"🔑 Nouvel onglet actif = {new_tab}")
    print(f"📍📍📍📍📍📍📍 URL = {driver.current_url}")

    # Attente optionnelle pour chargement
    for i in range(wait_seconds):
        print(f"⏳ Attente {i+1}/{wait_seconds}s")
        time.sleep(1)




    # dans l'ordre : 
    # initialiser data_champ 
    data_champ = { 
    "name": champion_descriptor["name"],
    }
    # Ajouter ces lignes avant d'itérer sur les rôles
    data_champ["matchup"] = {}   # ✅ initialisation pour éviter KeyError
    data_champ["synergy"] = {}  # ✅ initialisation pour éviter KeyError
    # collecter les données du champion dont role_champ, mettre ces données dans le dictionnaire data_champ
    lane_and_percentage = get_selected_played_lane(driver)
    # {"lane": None, "percentage": None}
    data_champ["role"] = lane_and_percentage["lane"]
    data_champ["role_play_ratio"] = lane_and_percentage["percentage"]
    general_data_for_champ_at_lane = collect_champion_lane_stats(driver)
    data_champ["tier"] = general_data_for_champ_at_lane["tier"]
    data_champ["rank"] = general_data_for_champ_at_lane["rank"]
    data_champ["winrate"] = general_data_for_champ_at_lane["winrate"]
    data_champ["pickrate"] = general_data_for_champ_at_lane["pickrate"]
    data_champ["banrate"] = general_data_for_champ_at_lane["banrate"]
    data_champ["nb_games_analyzed"] = general_data_for_champ_at_lane["games"]
    print(data_champ)
    # cliquer sur matchups
    click_matchup_or_synergy(driver, matchup=(param_synergy_or_matchup == "matchup"))
    roles = ["top", "jun", "mid", "adc", "sup"]

    # for role in roles:
    for i, role in enumerate(roles):
        # print(i, role)
        if not (role == param_lane):
            # print(f"⚠️ ⚠️ ⚠️Skipping matchup vs PAS même rôle ({role})")
            continue
        if not (param_synergy_or_matchup == "matchup"):
            if(param_lane == data_champ["role"]):
                # print(f"⚠️ ⚠️ ⚠️ ⚠️lane analysee est même que lane principale du champion, skipping ({role})")
                continue
        print(f"Collecte matchup vs rôle : {role}")
        # cliquer sur le bouton du rôle
        click_matchup_or_synergy_lane(driver, role)
        time.sleep(1)
        # cliquer sur "liste complète" UNE SEULE FOIS
        complet = click_liste_complete(driver)
        matchup = collect_matchup(driver, matchup=(param_synergy_or_matchup == "matchup"))
        # stocker
        colone = param_synergy_or_matchup
        data_champ[colone][role] = matchup

    print(data_champ)

    for i in range(1, wait_seconds - 2):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    print("🌍 URL champion :", driver.current_url)
    data_champ["url"] = driver.current_url  # (optionnel mais très utile)
    print("champ name = ", data_champ["name"])
    data_champ["collect_coherente"] = data_champ["name"].lower() in data_champ["url"].lower()






    # # Exemple : récupérer juste le nom du champion
    # data_champ = {
    #     "name": champion_descriptor.get("name"),
    #     "url": driver.current_url
    # }

    # Fermer l'onglet et revenir à la fenêtre principale
    print("🔙🔙🔙🔙🔙 Fermeture de l'onglet du champion et retour à la liste")
    driver.close()
    driver.switch_to.window(main_window)
    print("↩️ Retour à la liste des champions")

    return data_champ


In [13]:
def build_champion_dict(obj):
    """
    Transforme le JSON imbriqué en :
    {
        "lane_mode_elo_server_patch": [champions...],
        ...
    }
    """

    result = {}

    def walk(node, path_keys):
        # cas feuille = liste de champions
        if isinstance(node, list):
            if len(path_keys) == 4:
                lane_mode, elo, server, patch = path_keys
                key = f"{lane_mode}_{elo}_{server}_{patch}"
                result[key] = node
                print(f"✅ Liste champions trouvée → {key} ({len(node)} champions)")
            return

        if isinstance(node, dict):
            for k, v in node.items():
                walk(v, path_keys + [k])

    walk(obj, [])
    return result


In [ ]:
champion_dict = build_champion_dict(sauvegarde_champs_en_cours)

print(len(champion_dict))
print(list(champion_dict.keys())[:5])
for k, v in champion_dict.items():
    print(k, len(v))

In [14]:
import csv
import json

def save_champion_dict_to_csv(champion_dict, csv_path):
    """
    Enregistre champion_dict dans un CSV.
    Format :
        key ; json_list_of_champions
    """

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["key", "champions_json"])

        for key, champions in champion_dict.items():
            champions_json = json.dumps(champions, ensure_ascii=False)
            writer.writerow([key, champions_json])

    print(f"✅ champion_dict sauvegardé → {csv_path}")

def load_champion_dict_from_csv(csv_path):
    """
    Recharge le dictionnaire depuis le CSV produit ci-dessus.
    """

    champion_dict = {}

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = row["key"]
            champions = json.loads(row["champions_json"])
            champion_dict[key] = champions

    print(f"✅ champion_dict chargé ← {csv_path}")
    return champion_dict

# save_champion_dict_to_csv(champion_dict, "champions_dump_complet.csv")

champion_dict_bis = load_champion_dict_from_csv("champions_dump_complet.csv")

✅ champion_dict chargé ← champions_dump_complet.csv


In [17]:
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [16]:


options = webdriver.ChromeOptions()
options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [16]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
✅ Connecté à Chrome existant


In [15]:

def process_champion_dict_key(driver, champion_dict, key, ctx, min_idx=0, max_idx=None):
    """
    Traite uniquement la liste de champions correspondant à une clé spécifique du dictionnaire.

    Parameters
    ----------
    driver : WebDriver
    champion_dict : dict[str, list[champ]]
        Dictionnaire complet {clé: liste de champions}
    key : str
        Clé du dictionnaire à traiter
    ctx : dict
        Contexte contenant lane, mode, elo, server, patch, nom_csv
    min_idx : int
        Index du premier champion à traiter dans la liste
    max_idx : int | None
        Index du dernier champion à traiter dans la liste (non inclus)
    """

    if key not in champion_dict:
        print(f"⚠️ Clé '{key}' introuvable dans le dictionnaire.")
        return

    champ_list = champion_dict[key]
    if max_idx is None or max_idx > len(champ_list):
        max_idx = len(champ_list)

    print(f"\n==============================")
    print(f"🚀 Traitement clé '{key}' ({min_idx}-{max_idx})")
    print(f"📊 longueur de la liste: {len(champ_list)}")
    print(f"📦 {max_idx - min_idx} champions")
    print(f"==============================")

    ctx["all_champions_data"] = []

    for champ_dict_item in champ_list[min_idx:max_idx]:
        print("========================= Début traitement champion", champ_dict_item.get("name"))
        print(champ_dict_item)
        try:
            champ = open_champion_in_new_tab(
                driver,
                champ_dict_item,
                3,  # secondes d'attente
                ctx["lane"],
                ctx["mode"]
            )

            if champ is not None:
                ctx["all_champions_data"].append(champ)

        except Exception as e:
            print("❌ Erreur champion:", champ_dict_item.get("name"))
            print(e)
            continue

    # 👉 génération CSV après le traitement
    if ctx["all_champions_data"]:
        generate_csv_from_champions(
            ctx["all_champions_data"],
            ctx["elo"],
            ctx["server"],
            ctx["patch"],
            ctx["lane"],
            ctx["mode"],
            output_path=ctx["nom_csv"],
        )

        print("✅ CSV généré :", ctx["nom_csv"])
        print("📐 Nb lignes :", len(ctx["all_champions_data"]))

    else:
        print("⚠️ Aucun champion traité — CSV non généré")


In [21]:
champion_dict_bis = load_champion_dict_from_csv("champions_dump_complet.csv")

✅ champion_dict chargé ← champions_dump_complet.csv


In [28]:
# # lanes = ['top', 'jun', 'mid', 'adc', 'sup']
# # modes = ["matchup", "synergy"]

# # elos = [
# #     'Challenger', 'Grandmaster', 'Master+', 'Master',
# #     'Diamond+', 'Diamond',
# #     'Emerald+', 'Emerald',
# #     'Platinum+', 'Platinum',
# #     'Gold+', 'Gold',
# #     'Silver+', 'Bronze', 'Iron',
# #     'TOUT'
# # ]

# # servers = [
# #     'EUW', 'KR', 'NA', 'BR', 'EUNE', 'JP',
# #     'LAN', 'LAS', 'OCE', 'RU', 'TR', 'VN',
# #     'TOUT'
# # ]

# # patches = [
# #     '7days', '14days', '30days',
# #     '16.3', '16.2', '16.1',
# #     '15.24', '15.23', '15.22'
# # ]

# champion_dict = build_champion_dict(sauvegarde_champs_en_cours)

# min_idx = 1
# max_idx = 50  # Traiter les champions d'index 0 à 4
# key = "top_matchup_TOUT_TOUT_16.3"
# nom_csv = f"{max_idx}-{min_idx}_{key}.csv"
# ctx = {
#     "lane": "top",
#     "mode": "matchup",
#     "elo": "TOUT",
#     "server": "TOUT",
#     "patch": "16.3",
#     "nom_csv": nom_csv
# }

# # Traite seulement les champions 0 à 4
# process_champion_dict_key(driver, champion_dict, key, ctx, min_idx, max_idx)

# champion_dict = build_champion_dict(sauvegarde_champs_en_cours)

intervalle = 5
for i in range(135, 146, intervalle):  # Traite les champions par groupes de 50
    min_idx = i
    max_idx = min(i + intervalle, 219)  # Ne pas dépasser 212 champions
    key = "top_matchup_TOUT_TOUT_15.24"
    nom_csv = f"{max_idx}-{min_idx}_sup_matchup_TOUT_TOUT_15.24.csv"
    ctx = {
        "lane": "sup",
        "mode": "matchup",
        "elo": "TOUT",
        "server": "TOUT",
        "patch": "15.24",
        "nom_csv": nom_csv
    }

    # Traite seulement les champions 0 à 4
    process_champion_dict_key(driver, champion_dict_bis, key, ctx, min_idx, max_idx)





🚀 Traitement clé 'top_matchup_TOUT_TOUT_15.24' (135-140)
📊 longueur de la liste: 219
📦 5 champions
========================= Début traitement champion Graves
{'label': '136', 'numero': 21, 'name': 'Graves', 'url': 'https://dpm.lol/champions/Graves/build?lane=jungle&tier=all&platform=all&timeframe=15.24', 'scroll_page': 5250, 'scroll_table': 0}
OPEN CHAMPION --- DEBUT DE LA METHODE
lane a inspecter: sup
synergy ou matchup: matchup


🪟 Onglets AVANT = ['0560600859D991DA0B2F24FCD08AEB76']
🌍 tentative d' Ouverture champion '136' dans un nouvel onglet → https://dpm.lol/champions/Graves/build?lane=jungle&tier=all&platform=all&timeframe=15.24
🪟 Onglets APRES = ['0560600859D991DA0B2F24FCD08AEB76', 'A6388E8B9B7016278493703B732BE68B']
🔑 Nouvel onglet actif = A6388E8B9B7016278493703B732BE68B
📍📍📍📍📍📍📍 URL = https://dpm.lol/champions/Graves/build?lane=jungle&tier=all&platform=all&timeframe=15.24
⏳ Attente 1/3s
⏳ Attente 2/3s
⏳ Attente 3/3s

🟢 Détection de la played lane sélectionnée

🔁 PASS GLOBAL 1/10
🔍 2 containers candidats trouvés
🔍 5 boutons de lane trouvés
[Lane 0] classes = p-8
[Lane 1] classes = p-8 text-black-900 rounded-md bg-blue-200/90
⭐ Lane sélectionnée détectée (index 1)
🏷️🏷️🏷️ lane = jun
📊📊📊 percentage = 93.6%
🟢 RESULTAT VALIDE
collect_champion_lane_stats 🟢 STATS DE LANE --- Collecte des stats champion / lane
collect_champion_lane_stats 
🔁 Tentative stats 1/3
collect_champion_lane_stats ✅ Container stats trouvé

In [25]:
options = webdriver.ChromeOptions()
options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
# driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [26]:
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

In [24]:
driver.quit()